In [2]:
import requests
import os
from io import BytesIO
from zipfile import ZipFile

base_url = "http://gtfs.adelaidemetro.com.au/v1"
latest_version_number = "static/latest/version.txt"
latest_version_feed = "static/latest/google_transit.zip"

# binary data as response
request_headers={"Content-Type": "application/octet-stream"}

# Feed template
# practice: how to use string template + format
feed_url_template = "static/{version}/google_transit.zip"

# output
dest_path = 'destination'
os.makedirs(dest_path, exist_ok=True)

Read latest version

In [3]:
# 1. Find the latest feed version
# 2. Download latest GTFS feed version using request
# # Feed's URL

# 1. Find the latest feed version
latest_version_url = f"{base_url}/{latest_version_number}"
resp_version = requests.get(latest_version_url, headers=request_headers)
resp_version.raise_for_status()          # raise error if request failed

version = resp_version.text.strip()      # e.g. "20250101_01"
print(f"Latest GTFS version: {version}")

# 2. Download latest GTFS feed using that version
feed_url = f"{base_url}/{feed_url_template.format(version=version)}"
print(f"Downloading feed from: {feed_url}")

resp_feed = requests.get(feed_url, headers=request_headers)
resp_feed.raise_for_status()

# Option A: extract ZIP directly into destination folder
with ZipFile(BytesIO(resp_feed.content)) as zf:
    zf.extractall(dest_path)

print(f"Feed extracted to: {os.path.abspath(dest_path)}")

Latest GTFS version: 1605
Feed extracted to: c:\Users\huyen\DE-K01\data_engineering_practices\module_2_deep_dive_into_data_engineering\01_data_sources\03_api\destination


In [5]:
resp = requests.get(feed_url)
print(resp.headers)


{'Content-Type': 'application/octet-stream', 'Content-Length': '15075431', 'Connection': 'keep-alive', 'Date': 'Thu, 11 Dec 2025 17:13:47 GMT', 'Last-Modified': 'Thu, 11 Dec 2025 05:23:03 GMT', 'ETag': '"85ee6f071136c1a3187a82d097cce787-2"', 'x-amz-server-side-encryption': 'AES256', 'Content-Disposition': 'attachment; filename=google_transit.zip', 'x-amz-version-id': '2tALbW.h_SkQTjxk9ovWfaydbCpuAAsS', 'Accept-Ranges': 'bytes', 'Server': 'AmazonS3', 'X-Cache': 'Hit from cloudfront', 'Via': '1.1 f4d9e5aa78d9bbc69bc2a7f8ca614182.cloudfront.net (CloudFront)', 'X-Amz-Cf-Pop': 'LHR61-P2', 'X-Amz-Cf-Id': 'X4yrOr5IAmslADf_X0vvxUu-EixvmOA42SuUJvHAOgE6ZuCq1Vu07w==', 'Age': '18'}


Dynamic read multiple feed versions: 930 to 939

In [6]:
# HINTS
# 1. For loop the versions
# 2. Generate versioned URL
# 3. Request and handle response

import os
from io import BytesIO
from zipfile import ZipFile
import requests

# assume these are already defined above:
# base_url = "http://gtfs.adelaidemetro.com.au/v1"
# feed_url_template = "static/{version}/google_transit.zip"
# request_headers = {"Content-Type": "application/octet-stream"}
# dest_path = "destination"

os.makedirs(dest_path, exist_ok=True)

start_version = 930
end_version = 939

for version in range(start_version, end_version + 1):
    print(f"\n=== Processing version {version} ===")

    # 1. Generate versioned URL
    feed_url = f"{base_url}/{feed_url_template.format(version=version)}"
    print(f"Requesting: {feed_url}")

    # 2. Request and handle response
    resp = requests.get(feed_url, headers=request_headers)

    if resp.status_code == 200:
        print(f"Version {version}: OK, size = {len(resp.content)} bytes")

        # (Optional) extract each version into its own folder
        version_dest = os.path.join(dest_path, f"v_{version}")
        os.makedirs(version_dest, exist_ok=True)

        with ZipFile(BytesIO(resp.content)) as zf:
            zf.extractall(version_dest)

        print(f"→ Extracted to: {os.path.abspath(version_dest)}")

    elif resp.status_code == 404:
        print(f"Version {version}: NOT FOUND (404)")
    else:
        print(f"Version {version}: Error {resp.status_code}")



=== Processing version 930 ===
Requesting: http://gtfs.adelaidemetro.com.au/v1/static/930/google_transit.zip
Version 930: OK, size = 11876923 bytes
→ Extracted to: c:\Users\huyen\DE-K01\data_engineering_practices\module_2_deep_dive_into_data_engineering\01_data_sources\03_api\destination\v_930

=== Processing version 931 ===
Requesting: http://gtfs.adelaidemetro.com.au/v1/static/931/google_transit.zip
Version 931: OK, size = 11876998 bytes
→ Extracted to: c:\Users\huyen\DE-K01\data_engineering_practices\module_2_deep_dive_into_data_engineering\01_data_sources\03_api\destination\v_931

=== Processing version 932 ===
Requesting: http://gtfs.adelaidemetro.com.au/v1/static/932/google_transit.zip
Version 932: OK, size = 11781803 bytes
→ Extracted to: c:\Users\huyen\DE-K01\data_engineering_practices\module_2_deep_dive_into_data_engineering\01_data_sources\03_api\destination\v_932

=== Processing version 933 ===
Requesting: http://gtfs.adelaidemetro.com.au/v1/static/933/google_transit.zip
Ve

Same code - but in chunk Because in the chunked version, we saved the ZIP to disk first:Then extracted it.
Previously, you extracted directly from memory, so you never saw the ZIP file.

In [9]:
### same code - but in chunk

import os
from zipfile import ZipFile
import requests

# assume earlier:
# base_url = "http://gtfs.adelaidemetro.com.au/v1"
# feed_url_template = "static/{version}/google_transit.zip"
# request_headers = {"Content-Type": "application/octet-stream"}

dest_path = "destination_chunks"
os.makedirs(dest_path, exist_ok=True)

start_version = 930
end_version = 939
chunk_size = 1024 * 1024   # 1 MB

for version in range(start_version, end_version + 1):
    print(f"\n==============================")
    print(f" Processing version {version}")
    print(f"==============================")

    # 1. Build URL
    feed_url = f"{base_url}/{feed_url_template.format(version=version)}"
    print(f"[INFO] Requesting URL: {feed_url}")

    # 2. Stream request
    resp = requests.get(feed_url, headers=request_headers, stream=True)

    # Print STATUS CODE
    print(f"[STATUS] HTTP {resp.status_code}")

    # Handle 404 or other errors
    if resp.status_code == 404:
        print(f"[INFO] Version {version} NOT FOUND (404). Skipping.")
        continue
    elif resp.status_code != 200:
        print(f"[ERROR] Unexpected error {resp.status_code}. Skipping.")
        continue

    # 3. Download ZIP in chunks
    zip_path = os.path.join(dest_path, f"gtfs_{version}.zip")
    print(f"[INFO] Saving ZIP file to: {zip_path}")

    total_bytes = 0
    chunk_count = 0

    with open(zip_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                chunk_count += 1
                total_bytes += len(chunk)
                print(f" → Chunk {chunk_count} received ({len(chunk)} bytes)")

    print(f"[INFO] Download complete!")
    print(f"[INFO] Total chunks: {chunk_count}, total size: {total_bytes:,} bytes")

    # 4. Extract ZIP
    extract_folder = os.path.join(dest_path, f"extracted_v{version}")
    os.makedirs(extract_folder, exist_ok=True)

    print(f"[INFO] Extracting ZIP to: {extract_folder}")

    with ZipFile(zip_path) as zf:
        zf.extractall(extract_folder)

    print(f"[SUCCESS] Extracted version {version} into: {extract_folder}")




 Processing version 930
[INFO] Requesting URL: http://gtfs.adelaidemetro.com.au/v1/static/930/google_transit.zip
[STATUS] HTTP 200
[INFO] Saving ZIP file to: destination_chunks\gtfs_930.zip
 → Chunk 1 received (1048576 bytes)
 → Chunk 2 received (1048576 bytes)
 → Chunk 3 received (1048576 bytes)
 → Chunk 4 received (1048576 bytes)
 → Chunk 5 received (1048576 bytes)
 → Chunk 6 received (1048576 bytes)
 → Chunk 7 received (1048576 bytes)
 → Chunk 8 received (1048576 bytes)
 → Chunk 9 received (1048576 bytes)
 → Chunk 10 received (1048576 bytes)
 → Chunk 11 received (1048576 bytes)
 → Chunk 12 received (342587 bytes)
[INFO] Download complete!
[INFO] Total chunks: 12, total size: 11,876,923 bytes
[INFO] Extracting ZIP to: destination_chunks\extracted_v930
[SUCCESS] Extracted version 930 into: destination_chunks\extracted_v930

 Processing version 931
[INFO] Requesting URL: http://gtfs.adelaidemetro.com.au/v1/static/931/google_transit.zip
[STATUS] HTTP 200
[INFO] Saving ZIP file to: dest

Unzip feed versions & export files

In [15]:
# # using ZipFile to read zipfile
# # then extract all

# # Define the path to your zip file
# zip_file_path = 'destination/latest.zip' 

# # Define the directory where you want to extract the files (optional)
# # If not specified, files will be extracted to the current working directory
# extract_to_path = 'destination/latest/' 

# with ZipFile(zip_file_path, 'r') as zip_obj:
#     # Extract all files to the specified directory
#     zip_obj.extractall(extract_to_path)

import os
from zipfile import ZipFile

zip_folder = 'destination_chunks'        # folder that contains many .zip files
extract_base = 'destination_chunks/extracted_all'  # parent folder for extracted files

os.makedirs(extract_base, exist_ok=True)

# Loop through all files in the folder
for file in os.listdir(zip_folder):
    
    # Process only .zip files
    if file.lower().endswith('.zip'):
        zip_path = os.path.join(zip_folder, file)

        extract_to = os.path.join(extract_base, file.replace('.zip', ''))
        os.makedirs(extract_to, exist_ok=True)

        print(f"\n🔍 Extracting: {zip_path}")
        
        try:
            with ZipFile(zip_path, 'r') as zip_obj:
                zip_obj.extractall(extract_to)
            print(f"✅ Extracted to: {os.path.abspath(extract_to)}")

        except Exception as e:
            print(f"❌ Failed to extract {file}: {e}")



🔍 Extracting: destination_chunks\gtfs_930.zip
✅ Extracted to: c:\Users\huyen\DE-K01\data_engineering_practices\module_2_deep_dive_into_data_engineering\01_data_sources\03_api\destination_chunks\extracted_all\gtfs_930

🔍 Extracting: destination_chunks\gtfs_931.zip
✅ Extracted to: c:\Users\huyen\DE-K01\data_engineering_practices\module_2_deep_dive_into_data_engineering\01_data_sources\03_api\destination_chunks\extracted_all\gtfs_931

🔍 Extracting: destination_chunks\gtfs_932.zip
✅ Extracted to: c:\Users\huyen\DE-K01\data_engineering_practices\module_2_deep_dive_into_data_engineering\01_data_sources\03_api\destination_chunks\extracted_all\gtfs_932

🔍 Extracting: destination_chunks\gtfs_933.zip
✅ Extracted to: c:\Users\huyen\DE-K01\data_engineering_practices\module_2_deep_dive_into_data_engineering\01_data_sources\03_api\destination_chunks\extracted_all\gtfs_933

🔍 Extracting: destination_chunks\gtfs_934.zip
✅ Extracted to: c:\Users\huyen\DE-K01\data_engineering_practices\module_2_deep_di

Extra: extract zipfile from response.content

In [16]:
# HINTS
# Use BytesIO (file bytes in memory) + ZipFile

import requests
from zipfile import ZipFile
from io import BytesIO
import os

# Example ZIP URL (replace with any zip file URL)
zip_url = "http://gtfs.adelaidemetro.com.au/v1/static/latest/google_transit.zip"

output_folder = 'destination_chunks/content'
os.makedirs(output_folder, exist_ok=True)

# 1. Download ZIP as binary content (no saving to disk)
resp = requests.get(zip_url)
resp.raise_for_status()

# 2. Use BytesIO to treat bytes as a file-like object
zip_bytes = BytesIO(resp.content)

# 3. Extract ZIP in memory
with ZipFile(zip_bytes) as zip_obj:
    zip_obj.extractall(output_folder)

print(f"Extracted in-memory zip to: {os.path.abspath(output_folder)}")


Extracted in-memory zip to: c:\Users\huyen\DE-K01\data_engineering_practices\module_2_deep_dive_into_data_engineering\01_data_sources\03_api\destination_chunks\content
